In [ ]:
agg_func = {
    'rating_overall':    'mean',
    'rating_welfare':    'mean',
    'rating_worklife':   'mean',
    'rating_culture':    'mean',
    'rating_promotion':  'mean',
    'rating_management': 'mean',
    'rate_recommend':    'mean',
    'rate_ceo':          'mean',
    'rate_growth':       'mean',
    'tokenized_pros':       lambda x: " ".join(map(str, x)),
    'tokenized_cons':       lambda x: " ".join(map(str, x)),
    'tokenized_suggestion': lambda x: " ".join(map(str, x)),
    'company': 'count'
}

df_company = df.groupby('company').agg(agg_func)\
               .rename(columns={'company': 'review_count'}).reset_index()

# welfare_neg 바이그램 카운팅용 원문 합산 (토큰화 전 원본)
df_company['review_cons_raw'] = df.groupby('company')['review_cons'].apply(
    lambda x: " ".join(x.fillna(''))).values

# 평점 100점 변환
for col in ['overall', 'welfare', 'worklife', 'culture', 'promotion', 'management']:
    df_company[f'{col}_score'] = (df_company[f'rating_{col}'] / 5) * 100

# 신뢰도 문자열
def get_reliability_label(count):
    if count >= 30: return '매우 높음'
    elif count >= 10: return '높음'
    elif count >= 3: return '보통'
    else: return '낮음'

# 신뢰도 숫자 (통합팀 병합 후 ML에서 활용)
def get_reliability_score(count):
    if count >= 30: return 1.0
    elif count >= 10: return 0.8
    elif count >= 3: return 0.6
    else: return 0.3

df_company['review_reliability']       = df_company['review_count'].apply(get_reliability_label)
df_company['review_reliability_score'] = df_company['review_count'].apply(get_reliability_score)

# 채용공고팀과 병합할 기준 컬럼
def normalize_company(name):
    name = str(name)
    name = re.sub(r'\(주\)|㈜|주식회사\s*|\(유\)|\(재\)', '', name)
    return name.strip()

df_company['normalized_company_name'] = df_company['company'].apply(normalize_company)
df_company['review_company_name']     = df_company['company']

print(f"기업별 집계 완료: {len(df_company)}개 기업")

In [ ]:
def count_kw(text, words):
    tokens = str(text).split()
    return sum(1 for t in tokens if t in words)

# 긍정 키워드 사전
keyword_dicts_pos = {
    'welfare_pos':  ['복지', '성과급', '보너스', '식대', '자기계발'],
    'growth_pos':   ['성장', '교육', '커리어', '승진', '프로젝트'],
    'worklife_pos': ['워라밸', '유연근무', '재택', '연차'],
    'culture_pos':  ['수평문화', '팀워크', '소통', '동료'],
    'jobfit_pos':   ['전문성', '전문', '역할', '명확', '직무', '적성', '업무분장']
}

# 부정 키워드 사전 (welfare_neg 제외 — 바이그램 방식으로 별도 처리)
keyword_dicts_neg = {
    'growth_neg':   ['비체계', '미래전망', '직무전문성부족'],
    'worklife_neg': ['야근', '업무강도', '주말근무'],
    'culture_neg':  ['사내정치', '수직문화', '조직문제'],
    'jobfit_neg':   ['직무전문성부족', '비체계']
}

# welfare_neg 바이그램 패턴
# "연봉이 낮다", "복지가 없다"처럼 명사+부정형용사 조합을 원문에서 직접 탐지
welfare_neg_patterns = [
    r'연봉.{0,15}(낮은|낮다|낮아|낮고)',
    r'연봉.{0,15}(적은|적다|적어)',
    r'연봉.{0,10}동결',
    r'성과급.{0,10}(없다|없어|없는|안나)',
    r'복지.{0,10}(없다|없어|없는)',
    r'복지.{0,10}부족',
    r'급여.{0,10}(낮은|낮다|낮아)',
    r'급여.{0,10}(적은|적다|적어)',
    r'인상.{0,10}(없다|없어|없는|안)',
]

def count_bigram_patterns(text, patterns):
    if not text: return 0
    return sum(1 for pattern in patterns if re.search(pattern, str(text)))

# 긍정 카운팅: tokenized_pros 기준
for key, words in keyword_dicts_pos.items():
    df_company[f'{key}_count'] = df_company['tokenized_pros'].apply(
        lambda x: count_kw(x, words))

# 부정 카운팅: tokenized_cons + tokenized_suggestion 합산
for key, words in keyword_dicts_neg.items():
    df_company[f'{key}_count'] = (
        df_company['tokenized_cons'].apply(lambda x: count_kw(x, words)) +
        df_company['tokenized_suggestion'].apply(lambda x: count_kw(x, words))
    )

# welfare_neg 카운팅: 바이그램 방식으로 원문에서 탐지
df_company['welfare_neg_count'] = df_company['review_cons_raw'].apply(
    lambda x: count_bigram_patterns(x, welfare_neg_patterns))

print("키워드 카운팅 완료")
print(df_company[['company', 'welfare_neg_count', 'culture_neg_count', 'worklife_neg_count']].head(5))

In [ ]:
# 기본 리뷰 점수
# rate_recommend는 0~100 범위이므로 스케일 변환 없이 그대로 사용
def base_review_score(row):
    return (
        row['overall_score']    * 0.20 +
        row['welfare_score']    * 0.15 +
        row['worklife_score']   * 0.15 +
        row['culture_score']    * 0.15 +
        row['promotion_score']  * 0.15 +
        row['management_score'] * 0.10 +
        row['rate_recommend']   * 0.10
    )

# Q1 점수: 사용자가 중요하게 보는 항목의 점수를 Q1 점수로 사용
def q1_score(row, q1):
    mapping = {
        '1': row['welfare_score'],
        '2': (row['overall_score'] + row['rate_recommend']) / 2,
        '3': (row['promotion_score'] + row['rate_growth']) / 2,
        '4': row['worklife_score'],
        '5': (row['culture_score'] + row['management_score']) / 2
    }
    return mapping.get(q1, row['overall_score'])

# Q2 점수: 사용자가 피하고 싶은 위험 요인의 부정 키워드 수로 구간별 점수 부여
def q2_score(row, q2):
    col_map = {
        '1': 'welfare_neg_count',
        '2': 'growth_neg_count',
        '3': 'culture_neg_count',
        '4': 'jobfit_neg_count'
    }
    cnt = row.get(col_map.get(q2, 'culture_neg_count'), 0)
    if cnt == 0:     return 90
    elif cnt <= 2:   return 75
    elif cnt <= 5:   return 60
    else:            return 40

# Q3 점수: 문화 긍정/부정 키워드 비율로 업무방식 궁합 점수 산출
# 키워드 합계 3개 미만이면 데이터 부족으로 중간값(70점) 부여
def q3_score(row, q3):
    pos = row.get('culture_pos_count', 0)
    neg = row.get('culture_neg_count', 0)
    total = pos + neg
    if total < 3:
        return 70
    ratio = pos / total
    if ratio >= 0.7:   return 90
    elif ratio >= 0.5: return 75
    elif ratio >= 0.3: return 55
    else:              return 40

# 테스트용 고정값 — 실서비스에서 user_input으로 교체
USER_Q = {'Q1': '4', 'Q2': '3', 'Q3': '1'}

df_company['base_review_score']    = df_company.apply(base_review_score, axis=1).round(1)
df_company['q1_preference_score']  = df_company.apply(
    lambda r: q1_score(r, USER_Q['Q1']), axis=1).round(1)
df_company['q2_risk_score']        = df_company.apply(
    lambda r: q2_score(r, USER_Q['Q2']), axis=1)
df_company['q3_fit_score']         = df_company.apply(
    lambda r: q3_score(r, USER_Q['Q3']), axis=1)

# 최종 점수: 흐름 PDF 9단계 공식
df_company['review_final_score'] = (
    df_company['base_review_score']   * 0.40 +
    df_company['q1_preference_score'] * 0.25 +
    df_company['q2_risk_score']       * 0.20 +
    df_company['q3_fit_score']        * 0.15
).round(1)

# 등급: 데이터 평균(56점)을 고려해 70점 이상 추천으로 설정
def review_grade(score):
    if score >= 70:   return '추천'
    elif score >= 50: return '보류'
    else:             return '비추천'

# 요약 문장: 조건별 자연어 문장 조합
def review_summary(row):
    msgs = []
    if row['worklife_score']    >= 60: msgs.append("워라밸 평가가 좋은 편입니다.")
    if row['welfare_score']     >= 60: msgs.append("복지/처우 평가가 비교적 좋습니다.")
    if row['culture_neg_count'] >= 3:  msgs.append("조직문화 관련 부정 리뷰가 일부 확인됩니다.")
    if row['growth_neg_count']  >= 3:  msgs.append("성장 한계 관련 리뷰가 일부 확인됩니다.")
    if row['review_count']      < 3:   msgs.append("리뷰 수가 적어 해석에 주의가 필요합니다.")
    return " ".join(msgs) if msgs else "분석 참고용 리뷰입니다."

df_company['review_grade']   = df_company['review_final_score'].apply(review_grade)
df_company['review_summary'] = df_company.apply(review_summary, axis=1)

print("점수화 완료")
print(df_company[['company', 'base_review_score', 'review_final_score', 'review_grade']].head(10))

In [ ]:
final_cols = [
    'normalized_company_name',
    'review_company_name',
    'review_count',
    'review_reliability_score',
    'overall_score',
    'welfare_score',
    'worklife_score',
    'culture_score',
    'promotion_score',
    'management_score',
    'welfare_pos_count',
    'welfare_neg_count',
    'growth_pos_count',
    'growth_neg_count',
    'worklife_pos_count',
    'worklife_neg_count',
    'culture_pos_count',
    'culture_neg_count',
    'jobfit_pos_count',
    'jobfit_neg_count',
    'q1_preference_score',
    'q2_risk_score',
    'q3_fit_score',
    'review_final_score',
    'review_grade',
    'review_summary'
]

# 경로는 본인 환경에 맞게 수정
output_path = r"C:\Users\itwill\Desktop\프로젝트\잡플래닛_리뷰점수화.xlsx"
df_company[final_cols].to_excel(output_path, index=False)
print(f"저장 완료: {len(df_company)}개 기업, {len(final_cols)}개 컬럼")